In [3]:
# -*- coding: utf-8 -*-

# =====================================================================
# 기본_모형_AR_시간별_일반LAD.py
#
# 목적:
#   "기본 모형 AR"을 일반적인 시간별 AR(12) 방식으로 구현한다.
#
#   AR(12):
#
#       S_t = α
#           + β1*S_(t-1)
#           + β2*S_(t-2)
#           + ...
#           + β12*S_(t-12)
#
#   즉, 각 시점 t를 예측할 때 바로 직전 12시간의 발전량을 사용한다.
#
#   학습:
#       일반 LAD (Least Absolute Deviations)
#
#       min Σ |y_t - X_t β|
#
#       학습 중에는 예측값 [0,1] 제약을 걸지 않는다.
#
#   테스트:
#       예측값을 계산한 뒤에만 [0,1]로 clip한다.
#
#   중요:
#       기존 코드처럼
#
#           오늘 09시 ← 전날 09~20시
#           오늘 10시 ← 전날 09~20시
#           ...
#
#       와 같이 "전날 profile 전체"를 사용하는 방식이 아니다.
#
#       대신:
#
#           오늘 09시 ← 전날 20,19,...,09
#           오늘 10시 ← 오늘 09, 전날 20,19,...,10
#           오늘 11시 ← 오늘 10,09, 전날 20,19,...,11
#           ...
#
#       처럼 매 시간 직전 12개의 관측값을 사용한다.
#
# 데이터:
#   merged_for_simulation_z01.csv
#
# 시간:
#   Sydney local time
#   09:00 ~ 20:00 (하루 12시간)
#
# 학습:
#   2012-11-28 ~ 2013-09-23
#
# 테스트:
#   2013-09-24 ~ 2014-01-01
#
# =====================================================================


import os
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.optimize import linprog


# =====================================================================
# 0. 설정값
# =====================================================================


BASE_DIR = os.getcwd()      
MERGED_FILE = os.path.join(
    BASE_DIR,
    "merged_for_simulation_z01.csv"
)


# ---------------------------------------------------------------------
# 하루의 예측 대상 시간
# ---------------------------------------------------------------------

HOURS_PER_DAY = 12

LOCAL_HOUR_START = 9
LOCAL_HOUR_END = 21       # 21 미만 → 09~20시


# ---------------------------------------------------------------------
# 학습 / 테스트 기간
# ---------------------------------------------------------------------

TRAIN_START = pd.Timestamp("2012-11-28")
TRAIN_END = pd.Timestamp("2013-09-23")

TEST_START = pd.Timestamp("2013-09-24")
TEST_END = pd.Timestamp("2014-01-01")


# ---------------------------------------------------------------------
# AR에 필요한 최초 하루
#
# 학습 첫 시점인
#
# 2012-11-28 09:00
#
# 을 예측하려면 직전 12시간,
#
# 2012-11-27 09:00 ~ 20:00
#
# 가 필요하다.
# ---------------------------------------------------------------------

HISTORY_DATE = pd.Timestamp("2012-11-27")


# ---------------------------------------------------------------------
# 논문에서 사용하는 발전소 용량 및 시장 관련 parameter
# ---------------------------------------------------------------------

CAPACITY_MW = 30.0

DURATION_HOURS = 1.0

PENALTY_RATE = 0.5


# ---------------------------------------------------------------------
# 논문 Table 3 참고값
# ---------------------------------------------------------------------

PAPER_NRMSE = 34.76

PAPER_GAP = 15.04


# ---------------------------------------------------------------------
# 기존 bounded LAD 코드 결과
# 비교용
# ---------------------------------------------------------------------

BOUNDED_LAD_NRMSE = 38.783485

BOUNDED_LAD_GAP = 3.696468


# =====================================================================
# 1. 데이터 읽기
# =====================================================================

raw_table = pd.read_csv(
    MERGED_FILE
)


# local_date를 날짜형으로 변환
raw_table["local_date"] = pd.to_datetime(
    raw_table["local_date"]
)


# ---------------------------------------------------------------------
# Sydney local time 기준으로 09~20시만 사용
# ---------------------------------------------------------------------

is_daylight = (
    (raw_table["local_hour"] >= LOCAL_HOUR_START)
    &
    (raw_table["local_hour"] < LOCAL_HOUR_END)
)


daylight_table = raw_table[
    is_daylight
].copy()


# ---------------------------------------------------------------------
# local_hour를 0~11의 hour index로 변환
#
# 09시 → 0
# 10시 → 1
# ...
# 20시 → 11
# ---------------------------------------------------------------------

daylight_table["hour_idx"] = (
    daylight_table["local_hour"]
    - LOCAL_HOUR_START
)


# =====================================================================
# 2. History / Train / Test 분리
# =====================================================================


# ---------------------------------------------------------------------
# History
# ---------------------------------------------------------------------

is_history_date = (
    daylight_table["local_date"]
    == HISTORY_DATE
)

history_rows = daylight_table[
    is_history_date
].copy()


history_rows = history_rows.sort_values(
    "hour_idx"
)


# ---------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------

is_train_date = (
    (daylight_table["local_date"] >= TRAIN_START)
    &
    (daylight_table["local_date"] <= TRAIN_END)
)

train_rows = daylight_table[
    is_train_date
].copy()


train_rows = train_rows.sort_values(
    ["local_date", "hour_idx"]
)


# ---------------------------------------------------------------------
# Test
# ---------------------------------------------------------------------

is_test_date = (
    (daylight_table["local_date"] >= TEST_START)
    &
    (daylight_table["local_date"] <= TEST_END)
)

test_rows = daylight_table[
    is_test_date
].copy()


test_rows = test_rows.sort_values(
    ["local_date", "hour_idx"]
)


# ---------------------------------------------------------------------
# 데이터 개수 확인
# ---------------------------------------------------------------------

print(
    "이력 날짜:",
    HISTORY_DATE.date(),
    "행 수:",
    len(history_rows)
)

print(
    "학습 구간:",
    TRAIN_START.date(),
    "~",
    TRAIN_END.date(),
    "행 수:",
    len(train_rows)
)

print(
    "테스트 구간:",
    TEST_START.date(),
    "~",
    TEST_END.date(),
    "행 수:",
    len(test_rows)
)


# =====================================================================
# 3. 데이터를 (날짜 × 12시간) 배열로 변환
# =====================================================================


# =====================================================================
# 3-1. History
# =====================================================================

history_solar = np.zeros(
    (1, HOURS_PER_DAY)
)


row_counter = 0


for _, one_row in history_rows.iterrows():

    hour_position = (
        row_counter
        % HOURS_PER_DAY
    )

    history_solar[
        0,
        hour_position
    ] = one_row["solar_power"]

    row_counter += 1


# =====================================================================
# 3-2. Train
# =====================================================================


train_dates_sorted = sorted(
    train_rows["local_date"].unique()
)


n_train_days = len(
    train_dates_sorted
)


train_solar = np.zeros(
    (
        n_train_days,
        HOURS_PER_DAY
    )
)


row_counter = 0


for _, one_row in train_rows.iterrows():

    day_position = (
        row_counter
        // HOURS_PER_DAY
    )

    hour_position = (
        row_counter
        % HOURS_PER_DAY
    )

    train_solar[
        day_position,
        hour_position
    ] = one_row["solar_power"]

    row_counter += 1


# =====================================================================
# 3-3. Test
# =====================================================================


test_dates_sorted = sorted(
    test_rows["local_date"].unique()
)


n_test_days = len(
    test_dates_sorted
)


test_solar = np.zeros(
    (
        n_test_days,
        HOURS_PER_DAY
    )
)


test_da_price = np.zeros(
    (
        n_test_days,
        HOURS_PER_DAY
    )
)


test_rt_price = np.zeros(
    (
        n_test_days,
        HOURS_PER_DAY
    )
)


row_counter = 0


for _, one_row in test_rows.iterrows():

    day_position = (
        row_counter
        // HOURS_PER_DAY
    )

    hour_position = (
        row_counter
        % HOURS_PER_DAY
    )

    test_solar[
        day_position,
        hour_position
    ] = one_row["solar_power"]

    test_da_price[
        day_position,
        hour_position
    ] = one_row["da_price"]

    test_rt_price[
        day_position,
        hour_position
    ] = one_row["rt_price"]

    row_counter += 1


# =====================================================================
# 4. 시간별 AR(12) 학습 데이터 만들기
# =====================================================================
#
# 핵심:
#
# 기존 코드:
#
#   하루마다 12개 feature를 만들고
#   각 hour별로 별도의 regression을 수행
#
# 수정 코드:
#
#   전체 시간을 하나의 연속적인 time series로 만들고
#
#       X_t =
#       [1,
#        S_(t-1),
#        S_(t-2),
#        ...
#        S_(t-12)]
#
#   를 만들어 하나의 AR(12) regression을 수행한다.
#
# =====================================================================


# ---------------------------------------------------------------------
# History + Train을 하나의 연속적인 시간 series로 만든다.
#
# History:
#   2012-11-27 09~20
#
# Train:
#   2012-11-28 09~20
#   ...
#   2013-09-23 09~20
#
# 총:
#
#   301일 × 12시간
#
# ---------------------------------------------------------------------

history_and_train_solar = np.vstack(
    [
        history_solar,
        train_solar
    ]
)


# ---------------------------------------------------------------------
# 2차원 배열을 시간 순서의 1차원 배열로 변환
#
# 예:
#
# [Day1_09, Day1_10, ..., Day1_20,
#  Day2_09, Day2_10, ..., Day2_20,
#  ...]
#
# ---------------------------------------------------------------------

solar_series = (
    history_and_train_solar.flatten()
)


# ---------------------------------------------------------------------
# 전체 학습 시점 수
#
# Train:
#
#   300일 × 12시간 = 3600개
#
# ---------------------------------------------------------------------

n_ar_samples = (
    n_train_days
    * HOURS_PER_DAY
)


# ---------------------------------------------------------------------
# 절편(intercept) column
# ---------------------------------------------------------------------

ar_intercept_column = np.ones(
    (
        n_ar_samples,
        1
    )
)


# ---------------------------------------------------------------------
# AR(12) lag feature
#
# column 0 → S(t-1)
# column 1 → S(t-2)
# ...
# column 11 → S(t-12)
#
# ---------------------------------------------------------------------

ar_lag_features = np.zeros(
    (
        n_ar_samples,
        HOURS_PER_DAY
    )
)


# =====================================================================
# 각 시간 t마다 직전 12시간을 가져온다.
# =====================================================================


for t in range(n_ar_samples):


    # -----------------------------------------------------------------
    # solar_series에서 현재 학습 대상 위치
    #
    # history 하루가 앞에 있기 때문에
    # Train 첫 번째 시점은 index 12부터 시작한다.
    # -----------------------------------------------------------------

    current_position = (
        HOURS_PER_DAY
        + t
    )


    # -----------------------------------------------------------------
    # 직전 12시간 추출
    #
    # 원래:
    #
    #   [t-12, t-11, ..., t-2, t-1]
    #
    # AR coefficient가
    #
    #   β1 → t-1
    #   β2 → t-2
    #   ...
    #
    # 가 되도록 역순으로 만든다.
    #
    # 결과:
    #
    #   [t-1, t-2, ..., t-12]
    #
    # -----------------------------------------------------------------

    previous_12_values = (
        solar_series[
            current_position - HOURS_PER_DAY
            :
            current_position
        ][::-1]
    )


    # -----------------------------------------------------------------
    # AR feature matrix에 저장
    # -----------------------------------------------------------------

    ar_lag_features[t] = (
        previous_12_values
    )


# ---------------------------------------------------------------------
# 최종 AR 설계행렬
#
# X =
#
# [1, S(t-1), S(t-2), ..., S(t-12)]
#
# ---------------------------------------------------------------------

ar_design_matrix = np.hstack(
    [
        ar_intercept_column,
        ar_lag_features
    ]
)


# ---------------------------------------------------------------------
# AR target
#
# 현재 시점의 실제 발전량
#
# y = S(t)
# ---------------------------------------------------------------------

ar_target = solar_series[
    HOURS_PER_DAY:
]


# ---------------------------------------------------------------------
# 확인
# ---------------------------------------------------------------------

print()
print(
    "=== AR(12) 학습 데이터 ==="
)

print(
    "X shape:",
    ar_design_matrix.shape
)

print(
    "y shape:",
    ar_target.shape
)


# =====================================================================
# 5. 일반 LAD로 AR(12) 계수 추정
# =====================================================================
#
# 최소화:
#
#   min Σ |y_t - X_t β|
#
# 이를 LP로 변환한다.
#
# residual의 절댓값을 u_t라고 하면:
#
#   Xβ - u <= y
#  -Xβ - u <= -y
#
# 그리고:
#
#   min Σ u_t
#
# =====================================================================


n_features = (
    ar_design_matrix.shape[1]
)


# ---------------------------------------------------------------------
# 계수 저장
#
# [intercept,
#  beta1,
#  beta2,
#  ...,
#  beta12]
#
# ---------------------------------------------------------------------

coefficients = np.zeros(
    n_features
)


# ---------------------------------------------------------------------
# 희소행렬 변환
# ---------------------------------------------------------------------

X_sparse = sparse.csr_matrix(
    ar_design_matrix
)


# ---------------------------------------------------------------------
# 절대값을 표현하기 위한 보조변수 u
# ---------------------------------------------------------------------

identity_matrix = sparse.eye(
    n_ar_samples,
    format="csr"
)


# =====================================================================
# LAD 제약조건
# =====================================================================


# Xβ - u <= y

constraint_block_1 = sparse.hstack(
    [
        X_sparse,
        -identity_matrix
    ]
)


# -Xβ - u <= -y

constraint_block_2 = sparse.hstack(
    [
        -X_sparse,
        -identity_matrix
    ]
)


# 두 제약조건을 합친다.

all_constraints = sparse.vstack(
    [
        constraint_block_1,
        constraint_block_2
    ],
    format="csr"
)


# ---------------------------------------------------------------------
# 우변
# ---------------------------------------------------------------------

constraint_limits = np.concatenate(
    [
        ar_target,
        -ar_target
    ]
)


# =====================================================================
# 목적함수
# =====================================================================
#
# β에는 목적함수 계수 0
#
# u에는 1/n
#
# 따라서:
#
#   min (1/n) Σu
#
# = 평균절대오차 최소화
#
# =====================================================================


objective_coefficients = np.concatenate(
    [
        np.zeros(n_features),
        np.ones(n_ar_samples)
        / n_ar_samples
    ]
)


# =====================================================================
# 변수 bounds
# =====================================================================
#
# 일반 LAD이므로:
#
#   β → 제한 없음
#
#   u → u >= 0
#
# [0,1] prediction constraint는 학습 중 적용하지 않는다.
#
# =====================================================================


variable_bounds = (
    [(None, None)] * n_features
    +
    [(0.0, None)] * n_ar_samples
)


# =====================================================================
# Linear Programming
# =====================================================================


lp_result = linprog(
    objective_coefficients,
    A_ub=all_constraints,
    b_ub=constraint_limits,
    bounds=variable_bounds,
    method="highs"
)


# ---------------------------------------------------------------------
# 성공 여부 확인
# ---------------------------------------------------------------------

print()
print(
    "=== AR(12) LAD 회귀 결과 ==="
)

print(
    "LP 성공 여부:",
    lp_result.success
)


if not lp_result.success:

    print(
        "LP 실패 메시지:",
        lp_result.message
    )

    raise RuntimeError(
        "AR(12) LAD regression failed."
    )


# ---------------------------------------------------------------------
# 회귀계수 추출
# ---------------------------------------------------------------------

coefficients = (
    lp_result.x[:n_features]
)


# ---------------------------------------------------------------------
# 회귀계수 출력
# ---------------------------------------------------------------------

print()
print(
    "=== AR(12) coefficients ==="
)

print(
    f"intercept = {coefficients[0]:.10f}"
)


for lag in range(1, HOURS_PER_DAY + 1):

    print(
        f"beta_{lag:02d} "
        f"(t-{lag}) = "
        f"{coefficients[lag]:.10f}"
    )


# =====================================================================
# 6. 테스트: Rolling one-step-ahead AR(12)
# =====================================================================
#
# 매우 중요:
#
# 테스트에서는 각 시간마다 직전 12시간의 실제 관측값을 사용한다.
#
# 예:
#
# 오늘 09시 예측:
#
#   전날 20,19,...,09
#
# 오늘 10시 예측:
#
#   오늘 09,
#   전날 20,19,...,10
#
# 오늘 11시 예측:
#
#   오늘 10,09,
#   전날 20,19,...,11
#
# ...
#
# 이렇게 매 시간 한 칸씩 rolling한다.
#
# ---------------------------------------------------------------------
#
# 여기서는 실제 발전량이 관측되었다고 가정하고 다음 시간 예측에
# 실제값을 사용한다.
#
# 따라서 "one-step-ahead forecasting"이다.
#
# =====================================================================


test_forecast = np.zeros(
    (
        n_test_days,
        HOURS_PER_DAY
    )
)


# ---------------------------------------------------------------------
# 테스트 시작 직전까지의 실제 발전량을 history로 준비
#
# 마지막 학습일:
#
# 2013-09-23 09~20
#
# ---------------------------------------------------------------------

rolling_history = list(
    train_solar.flatten()
)


# =====================================================================
# 테스트 100일
# =====================================================================


for day_index in range(n_test_days):


    # ---------------------------------------------------------------
    # 하루의 12시간
    # ---------------------------------------------------------------

    for hour in range(HOURS_PER_DAY):


        # -----------------------------------------------------------
        # 현재 시점 직전 12시간
        #
        # rolling_history의 마지막 12개:
        #
        # [t-12, ..., t-2, t-1]
        #
        # 이를 뒤집어서:
        #
        # [t-1, t-2, ..., t-12]
        #
        # 로 만든다.
        # -----------------------------------------------------------

        previous_12_values = np.array(
            rolling_history[-HOURS_PER_DAY:]
        )[::-1]


        # -----------------------------------------------------------
        # [intercept, t-1, t-2, ..., t-12]
        # -----------------------------------------------------------

        feature_vector = np.concatenate(
            [
                [1.0],
                previous_12_values
            ]
        )


        # -----------------------------------------------------------
        # AR(12) 예측
        #
        # Ŝ_t = X_t β
        # -----------------------------------------------------------

        raw_prediction = np.dot(
            coefficients,
            feature_vector
        )


        # -----------------------------------------------------------
        # 학습 단계에서는 [0,1] 제약이 없었다.
        #
        # 테스트에서만 물리적으로 가능한 범위로 clip한다.
        # -----------------------------------------------------------

        clipped_prediction = min(
            max(
                raw_prediction,
                0.0
            ),
            1.0
        )


        # -----------------------------------------------------------
        # 예측값 저장
        # -----------------------------------------------------------

        test_forecast[
            day_index,
            hour
        ] = clipped_prediction


        # -----------------------------------------------------------
        # 다음 시간 예측을 위해
        # 실제 관측값을 history에 추가한다.
        #
        # 중요:
        #
        # 여기서는 예측값이 아니라 실제값을 넣는다.
        #
        # 따라서 one-step-ahead forecasting이다.
        # -----------------------------------------------------------

        actual_value = test_solar[
            day_index,
            hour
        ]


        rolling_history.append(
            actual_value
        )


# =====================================================================
# 7. nRMSE 계산
# =====================================================================


actual_flat = (
    test_solar.flatten()
)


predicted_flat = (
    test_forecast.flatten()
)


# ---------------------------------------------------------------------
# MSE
# ---------------------------------------------------------------------

sum_of_squared_error = 0.0


for i in range(
    len(actual_flat)
):

    error_i = (
        actual_flat[i]
        -
        predicted_flat[i]
    )

    sum_of_squared_error += (
        error_i * error_i
    )


mean_squared_error = (
    sum_of_squared_error
    /
    len(actual_flat)
)


# ---------------------------------------------------------------------
# RMSE
# ---------------------------------------------------------------------

rmse_value = (
    mean_squared_error ** 0.5
)


# ---------------------------------------------------------------------
# 실제값 평균
# ---------------------------------------------------------------------

sum_of_actual = 0.0


for i in range(
    len(actual_flat)
):

    sum_of_actual += (
        actual_flat[i]
    )


average_actual = (
    sum_of_actual
    /
    len(actual_flat)
)


# ---------------------------------------------------------------------
# nRMSE
# ---------------------------------------------------------------------

nrmse_percent = (
    100.0
    *
    rmse_value
    /
    average_actual
)


# =====================================================================
# 8. Optimality Gap 계산
# =====================================================================
#
# 논문의 Eq.(1a) profit 구조:
#
#   DA revenue
#   + RT revenue for surplus
#   - penalty for shortage
#
# =====================================================================


da_flat = (
    test_da_price.flatten()
)


rt_flat = (
    test_rt_price.flatten()
)


sum_of_realized_profit = 0.0

sum_of_oracle_profit = 0.0


# =====================================================================
# 각 테스트 시간별 계산
# =====================================================================


for i in range(
    len(actual_flat)
):


    # -----------------------------------------------------------------
    # 실제 발전량
    # -----------------------------------------------------------------

    actual_i = (
        actual_flat[i]
    )


    # -----------------------------------------------------------------
    # AR 예측값
    #
    # = DA 시장 commitment
    # -----------------------------------------------------------------

    commitment_i = (
        predicted_flat[i]
    )


    # -----------------------------------------------------------------
    # 가격
    # -----------------------------------------------------------------

    da_i = (
        da_flat[i]
    )

    rt_i = (
        rt_flat[i]
    )


    # -----------------------------------------------------------------
    # shortage penalty
    #
    # DA 가격의 50%
    # -----------------------------------------------------------------

    penalty_cost_i = (
        PENALTY_RATE
        *
        da_i
    )


    # -----------------------------------------------------------------
    # 실제 발전량 - 약정량
    # -----------------------------------------------------------------

    mismatch_i = (
        actual_i
        -
        commitment_i
    )


    # -----------------------------------------------------------------
    # surplus
    # -----------------------------------------------------------------

    surplus_i = max(
        mismatch_i,
        0.0
    )


    # -----------------------------------------------------------------
    # shortage
    # ---------------------------------------------------------------------

    shortage_i = max(
        -mismatch_i,
        0.0
    )


    # =================================================================
    # 실제 AR 기반 profit
    # =================================================================

    realized_profit_i = (
        CAPACITY_MW
        *
        DURATION_HOURS
        *
        (
            da_i * commitment_i
            +
            rt_i * surplus_i
            -
            penalty_cost_i * shortage_i
        )
    )


    sum_of_realized_profit += (
        realized_profit_i
    )


    # =================================================================
    # Oracle
    # =================================================================
    #
    # Oracle에서는 실제 발전량을 알고 있다고 가정한다.
    #
    # commitment = 0
    #
    # 또는
    #
    # commitment = actual generation
    #
    # 두 경우 중 더 높은 profit을 선택한다.
    # =================================================================


    profit_if_commit_zero = (
        CAPACITY_MW
        *
        DURATION_HOURS
        *
        (
            rt_i
            *
            actual_i
        )
    )


    profit_if_commit_actual = (
        CAPACITY_MW
        *
        DURATION_HOURS
        *
        (
            da_i
            *
            actual_i
        )
    )


    oracle_profit_i = max(
        profit_if_commit_zero,
        profit_if_commit_actual
    )


    sum_of_oracle_profit += (
        oracle_profit_i
    )


# =====================================================================
# Optimality Gap
# =====================================================================


optimality_gap_percent = (
    100.0
    *
    (
        sum_of_oracle_profit
        -
        sum_of_realized_profit
    )
    /
    sum_of_oracle_profit
)


# =====================================================================
# 9. 결과 출력
# =====================================================================


print()

print(
    "============================================================"
)

print(
    " 기본 모형 AR(12) - 일반 LAD"
)

print(
    "============================================================"
)


print()

print(
    f"nRMSE = "
    f"{nrmse_percent:.6f} %"
)


print(
    f"optimality gap = "
    f"{optimality_gap_percent:.6f} %"
)


print()

print(
    "=== 논문 / 기존 bounded LAD / 현재 모델 비교 ==="
)


print()

print(
    "| 회귀 방식 | nRMSE | optimality gap |"
)

print(
    "|---|---:|---:|"
)


print(
    f"| 논문 기본 AR | "
    f"{PAPER_NRMSE:.2f}% | "
    f"{PAPER_GAP:.2f}% |"
)


print(
    f"| bounded LAD | "
    f"{BOUNDED_LAD_NRMSE:.6f}% | "
    f"{BOUNDED_LAD_GAP:.6f}% |"
)


print(
    f"| **시간별 AR(12) + 일반 LAD** | "
    f"**{nrmse_percent:.6f}%** | "
    f"**{optimality_gap_percent:.6f}%** |"
)


# =====================================================================
# 10. 추가 확인용 출력
# =====================================================================


print()

print(
    "=== 모델 구조 ==="
)

print(
    "S(t) = α + "
    "β1*S(t-1) + "
    "β2*S(t-2) + "
    "β3*S(t-3) + "
    "... + "
    "β12*S(t-12)"
)


print()

print(
    "학습 표본 수:",
    n_ar_samples
)


print(
    "AR lag 수:",
    HOURS_PER_DAY
)


print(
    "예측 방식:",
    "Rolling one-step-ahead"
)


print(
    "학습 loss:",
    "LAD (Mean Absolute Error)"
)


print(
    "학습 중 [0,1] 제약:",
    "없음"
)


print(
    "테스트 후 [0,1] clip:",
    "적용"
)

이력 날짜: 2012-11-27 행 수: 12
학습 구간: 2012-11-28 ~ 2013-09-23 행 수: 3600
테스트 구간: 2013-09-24 ~ 2014-01-01 행 수: 1200

=== AR(12) 학습 데이터 ===
X shape: (3600, 13)
y shape: (3600,)

=== AR(12) LAD 회귀 결과 ===
LP 성공 여부: True

=== AR(12) coefficients ===
intercept = 0.0061875433
beta_01 (t-1) = 0.9160329723
beta_02 (t-2) = -0.1673259986
beta_03 (t-3) = -0.0442830728
beta_04 (t-4) = -0.0152890881
beta_05 (t-5) = -0.0399499850
beta_06 (t-6) = -0.0266147657
beta_07 (t-7) = 0.0010664523
beta_08 (t-8) = 0.0183738121
beta_09 (t-9) = 0.0562475105
beta_10 (t-10) = -0.0017117353
beta_11 (t-11) = 0.0722808687
beta_12 (t-12) = 0.1642834944

 기본 모형 AR(12) - 일반 LAD

nRMSE = 33.720521 %
optimality gap = 6.266887 %

=== 논문 / 기존 bounded LAD / 현재 모델 비교 ===

| 회귀 방식 | nRMSE | optimality gap |
|---|---:|---:|
| 논문 기본 AR | 34.76% | 15.04% |
| bounded LAD | 38.783485% | 3.696468% |
| **시간별 AR(12) + 일반 LAD** | **33.720521%** | **6.266887%** |

=== 모델 구조 ===
S(t) = α + β1*S(t-1) + β2*S(t-2) + β3*S(t-3) + ... + β12*S(t-12)

